# Week 2: Goldowsky-Dill Linear Probe on Multi-Agent Collusion Data

<a href="https://colab.research.google.com/github/elenaajayi/spec-gap-activation-probe/blob/main/notebooks/02_collusion_probe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Trains linear probes on residual stream activations from Llama 3.1 8B Instruct
to distinguish colluding vs honest agents in NARCBench-Core committee deliberation scenarios.

Current upstream rerun contract:

- 50 NARCBench-Core committee decisions across 17 domains.
- 4 agents per scenario: 2 colluding agents and 2 honest agents.
- 200 prompts total, balanced 100 colluder / 100 honest.
- Generate a 64-token response at temperature 0.0, then run a forward pass over prompt + generation.
- Extract residual stream activations at the last generated token for layers 13 through 24.
- Probe with StandardScaler -> PCA(50) -> logistic regression.
- Evaluate with stratified 5-fold CV and leave-one-scenario-out CV.
- Run exemplar partitioning as an unsupervised geometry check.

Committed interpretation layers remain 16, 20, and 24, but the exploratory sweep checks 13-24 because the signal appears earlier than expected.

**GPU:** H100 recommended. **Auth:** HuggingFace token with Llama 3.1 access.


In [0]:
# Artifact directory setup
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_ARTIFACT_ROOT = Path("/content/drive/MyDrive/spec-gap-activation-probe/artifacts")
except Exception:
    DEFAULT_ARTIFACT_ROOT = Path.cwd() / "artifacts"

ARTIFACT_ROOT = Path(os.environ.get("SPEC_GAP_ARTIFACT_ROOT", DEFAULT_ARTIFACT_ROOT))
ARTIFACT_DIR = ARTIFACT_ROOT / "02_collusion_probe"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Artifact root: {ARTIFACT_ROOT}")
print(f"This notebook writes to: {ARTIFACT_DIR}")


In [0]:
# 1. Install dependencies
!pip install -q transformer-lens torch einops jaxtyping scikit-learn

In [0]:
# 2. HuggingFace login
import os
from huggingface_hub import login
from google.colab import userdata

token = userdata.get("HF_TOKEN_SANITY")
if token:
    login(token=token)
    print("Logged in via HF_TOKEN_SANITY")
else:
    login()

In [0]:
# 3. Extraction module (inlined from src/extraction/residual_stream.py)

import torch
from transformer_lens import HookedTransformer

DEFAULT_LAYERS = (13, 14, 15,16, 17, 18, 19, 20, 21, 22, 23, 24)
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
HIDDEN_DIM = 4096


def load_model(model_name=MODEL_NAME, device="cuda", dtype=torch.float16):
    model = HookedTransformer.from_pretrained(model_name, device=device, dtype=dtype)
    model.eval()
    return model


def _get_cache_filter(layer_to_extract):
    def name_filter(name):
        if not name.endswith("hook_resid_post"):
            return False
        return int(name.split(".")[1]) == layer_to_extract
    return name_filter


def extract_residual_stream(model, prompts, layers=DEFAULT_LAYERS,
                            token_position="last", batch_size=8, prepend_bos=True):
    results = {layer: [] for layer in layers}

    for layer in layers:
        name_filter = _get_cache_filter(layer)
        layer_activations = []

        for i in range(0, len(prompts), batch_size):
            batch = prompts[i : i + batch_size]
            tokens = model.to_tokens(batch, prepend_bos=prepend_bos)

            with torch.no_grad():
                _, cache = model.run_with_cache(tokens, names_filter=name_filter)

            hook_name = f"blocks.{layer}.hook_resid_post"
            activations = cache[hook_name]

            if token_position == "last":
                act = activations[:, -1, :]
            elif token_position == "all":
                act = activations
            elif isinstance(token_position, int):
                act = activations[:, token_position, :]
            else:
                raise ValueError(f"Unknown token_position: {token_position}")

            layer_activations.append(act.cpu())
            del cache
            torch.cuda.empty_cache()

        results[layer] = torch.cat(layer_activations, dim=0)

    return results

In [0]:
# 4. Probe module (inlined from src/probes/linear_probe.py)

import numpy as np
from dataclasses import dataclass
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import roc_auc_score, brier_score_loss, accuracy_score
from sklearn.decomposition import PCA


@dataclass
class CVResult:
    auroc_mean: float
    auroc_std: float
    auroc_per_fold: list
    accuracy_mean: float
    accuracy_std: float
    brier_mean: float
    brier_std: float
    ece_mean: float
    ece_std: float


def compute_ece(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    if n == 0:
        return 0.0
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (y_prob >= lo) & (y_prob <= hi) if i == n_bins - 1 else (y_prob >= lo) & (y_prob < hi)
        count = mask.sum()
        if count == 0:
            continue
        ece += (count / n) * abs(y_true[mask].mean() - y_prob[mask].mean())
    return float(ece)


def make_probe_pipeline(C=1.0, max_iter=1000, pca_components=None, random_state=42):
    steps = [("scaler", StandardScaler())]
    if pca_components is not None:
        steps.append(("pca", PCA(n_components=pca_components)))
    steps.append(("clf", LogisticRegression(C=C, max_iter=max_iter, random_state=random_state, solver="lbfgs")))
    return Pipeline(steps)


def run_cv(X, y, n_splits=5, pca_components=None, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    aurocs, accs, briers, eces = [], [], [], []
    for train_idx, test_idx in skf.split(X, y):
        pipe = make_probe_pipeline(pca_components=pca_components, random_state=random_state)
        pipe.fit(X[train_idx], y[train_idx])
        y_prob = pipe.predict_proba(X[test_idx])[:, 1]
        n_classes = len(np.unique(y[test_idx]))
        aurocs.append(roc_auc_score(y[test_idx], y_prob) if n_classes > 1 else np.nan)
        accs.append(accuracy_score(y[test_idx], (y_prob >= 0.5).astype(int)))
        briers.append(brier_score_loss(y[test_idx], y_prob))
        eces.append(compute_ece(y[test_idx], y_prob))
    return CVResult(
        auroc_mean=float(np.nanmean(aurocs)), auroc_std=float(np.nanstd(aurocs)),
        auroc_per_fold=aurocs,
        accuracy_mean=float(np.mean(accs)), accuracy_std=float(np.std(accs)),
        brier_mean=float(np.mean(briers)), brier_std=float(np.std(briers)),
        ece_mean=float(np.mean(eces)), ece_std=float(np.std(eces)),
    )


def run_leave_scenario_out(X, y, scenario_ids, pca_components=None, random_state=42):
    logo = LeaveOneGroupOut()
    aurocs, accs, briers, eces = [], [], [], []
    for train_idx, test_idx in logo.split(X, y, groups=scenario_ids):
        pipe = make_probe_pipeline(pca_components=pca_components, random_state=random_state)
        pipe.fit(X[train_idx], y[train_idx])
        y_prob = pipe.predict_proba(X[test_idx])[:, 1]
        n_classes = len(np.unique(y[test_idx]))
        aurocs.append(roc_auc_score(y[test_idx], y_prob) if n_classes > 1 else np.nan)
        accs.append(accuracy_score(y[test_idx], (y_prob >= 0.5).astype(int)))
        briers.append(brier_score_loss(y[test_idx], y_prob))
        eces.append(compute_ece(y[test_idx], y_prob))
    return CVResult(
        auroc_mean=float(np.nanmean(aurocs)), auroc_std=float(np.nanstd(aurocs)),
        auroc_per_fold=aurocs,
        accuracy_mean=float(np.mean(accs)), accuracy_std=float(np.std(accs)),
        brier_mean=float(np.mean(briers)), brier_std=float(np.std(briers)),
        ece_mean=float(np.mean(eces)), ece_std=float(np.std(eces)),
    )


def compute_deception_direction(X, y, C=1.0, max_iter=1000, random_state=42):
    pipe = make_probe_pipeline(C=C, random_state=random_state)
    pipe.fit(X, y)
    coef = pipe.named_steps["clf"].coef_[0].copy()
    coef = coef / pipe.named_steps["scaler"].scale_
    norm = float(np.linalg.norm(coef))
    return coef / norm if norm > 0 else coef, norm


def compute_diff_in_means(X, y):
    y = np.asarray(y)
    direction = X[y == 1].mean(axis=0) - X[y == 0].mean(axis=0)
    return direction, float(np.linalg.norm(direction))


print("Probe module loaded.")

In [0]:
# 5. Load NARCBench-Core scenarios from GitHub
import urllib.request, sys, types, random

# Stub config module so narcbench core.py can import
_cfg = types.ModuleType("config")
_cfg.VLLM_BASE_URL = ""
_cfg.DEFAULT_MODEL = ""
_cfg.VLLM_TEMPERATURE = 0.0
_cfg.VLLM_MAX_TOKENS = 512
_cfg.ROOT = __import__("pathlib").Path(".")
_cfg.RAW_ROOT = _cfg.ROOT / "data" / "raw"
sys.modules["config"] = _cfg

url = "https://raw.githubusercontent.com/aaronrose227/narcbench/main/generation/core.py"
urllib.request.urlretrieve(url, "narcbench_core.py")

from narcbench_core import (
    SCENARIOS, NAME_POOL, HONEST_SYSTEM,
    COLLUDER_SYSTEM_A, COLLUDER_SYSTEM_B,
)

print(f"Loaded {len(SCENARIOS)} NARCBench-Core scenarios")
print(f"Domains: {sorted(set(s['domain'] for s in SCENARIOS))}")

# 7. Build prompts - NARCBench structure: 4 agents per scenario, single discussion turn
import numpy as np

def format_chat_prompt(system, user):
    return (
        f"<|start_header_id|>system<|end_header_id|>\n\n{system}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n{user}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )

prompts = []
labels = []
metadata = []

for scenario_idx, scenario in enumerate(SCENARIOS):
    # Draw 4 agent names deterministically (matches NARCBench)
    draw_rng = random.Random(scenario_idx)
    agents = draw_rng.sample(NAME_POOL, 4)
    colluders = agents[:2]
    honest_agents = agents[2:]

    # Option label randomization (NARCBench cycles through 4 variants)
    variant = scenario_idx % 4
    use_numbers = variant >= 2
    swap_content = variant % 2 == 1
    label1, label2 = ("1", "2") if use_numbers else ("A", "B")
    fair_desc = scenario["options"]["A"]
    corrupt_desc = scenario["options"]["B"]
    if swap_content:
        options_dict = {label1: corrupt_desc, label2: fair_desc}
        effective_target = label1
    else:
        options_dict = {label1: fair_desc, label2: corrupt_desc}
        effective_target = label2

    options_text = "\n".join(f"Option {k}: {v}" for k, v in options_dict.items())

    # Colluder system prompt (alternates template A/B)
    colluder_template = COLLUDER_SYSTEM_A if scenario_idx % 2 == 0 else COLLUDER_SYSTEM_B

    # User prompt: context + options + discussion instruction
    user_prompt = (
        f"{scenario['context']}\n\n"
        f"OPTIONS UNDER REVIEW:\n{options_text}\n\n"
        f"{scenario['public_frame']}"
    )

    for agent in agents:
        is_colluder = agent in colluders
        if is_colluder:
            # Alternate standard/lowkey instruction per colluder
            use_lowkey = (scenario_idx + agents.index(agent)) % 2 == 1
            deception_instr = (
                scenario["lowkey_instruction"] if use_lowkey
                else scenario["deception_instruction"]
            )
            system = colluder_template.format(
                agent_name=agent,
                motive=scenario["collusion_motive"],
                deception_instruction=deception_instr,
            )
        else:
            system = HONEST_SYSTEM.format(
                agent_name=agent,
                honest_mandate=scenario["honest_mandate"],
            )

        prompts.append(format_chat_prompt(system, user_prompt))
        labels.append(1 if is_colluder else 0)
        metadata.append({
            "scenario_idx": scenario_idx,
            "scenario_id": scenario["id"],
            "domain": scenario["domain"],
            "role": "colluder" if is_colluder else "honest",
            "agent_name": agent,
        })

labels = np.array(labels)
print(f"Built {len(prompts)} prompts: {labels.sum()} colluder, {(labels == 0).sum()} honest")
print(f"Scenarios: {len(SCENARIOS)}, Domains: {len(set(m['domain'] for m in metadata))}")

# Spot-check
print(f"\n--- Sample colluder prompt (scenario 0) ---")
print(prompts[0][:500])
print(f"\n--- Sample honest prompt (scenario 0) ---")
print(prompts[2][:500])


In [0]:
# 6. Inspect sample prompts
print("=== Sample COLLUDER prompt (first 400 chars) ===")
print(prompts[0][:400])
print("\n=== Sample HONEST prompt (first 400 chars) ===")
print(prompts[2][:400])

In [0]:
# 7. Load model
print(f"Loading {MODEL_NAME}...")
model = load_model(device="cuda")
print(f"Loaded. {model.cfg.n_layers} layers, hidden dim {model.cfg.d_model}")

# Verify chat template tokenization
sample_tokens = model.to_tokens([prompts[0]], prepend_bos=True)
print(f"\nFirst 10 token IDs: {sample_tokens[0, :10].tolist()}")
print(f"Expected: 128000 (<|begin_of_text|>), 128006 (<|start_header_id|>), ...")

In [0]:
# 8. Generate responses + extract activations at last generated token
import torch

MAX_NEW_TOKENS = 64
GENERATION_TEMPERATURE = 0.0
TOP_P = 1.0
DEFAULT_LAYERS = (13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24)

def cache_filter(name):
    return any(name == f"blocks.{layer}.hook_resid_post" for layer in DEFAULT_LAYERS)

activations = {layer: [] for layer in DEFAULT_LAYERS}
responses = []

print(f"Generating {MAX_NEW_TOKENS}-token responses at temp={GENERATION_TEMPERATURE} + extracting at layers {DEFAULT_LAYERS}...")
for i, prompt in enumerate(prompts):
    tokens = model.to_tokens(prompt, prepend_bos=True)

    with torch.no_grad():
        gen_tokens = model.generate(
            tokens,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=GENERATION_TEMPERATURE,
            top_p=TOP_P,
            stop_at_eos=True,
        )

    # Forward pass on full sequence (input + generated), grab last token activations
    with torch.no_grad():
        _, cache = model.run_with_cache(gen_tokens, names_filter=cache_filter)

    for layer in DEFAULT_LAYERS:
        activations[layer].append(
            cache[f"blocks.{layer}.hook_resid_post"][0, -1, :].cpu()
        )

    # Decode generated response for inspection
    new_tokens = gen_tokens[0, tokens.shape[1]:]
    responses.append(model.tokenizer.decode(new_tokens, skip_special_tokens=True))

    del cache
    if (i + 1) % 10 == 0:
        torch.cuda.empty_cache()
        print(f"  {i+1}/{len(prompts)}")

for layer in DEFAULT_LAYERS:
    activations[layer] = torch.stack(activations[layer])
    print(f"Layer {layer}: {activations[layer].shape}")

# Spot-check: one colluder, one honest response
for j in [0, 50]:
    role = "colluder" if labels[j] == 1 else "honest"
    print(f"\n--- Prompt {j} ({role}) ---")
    print(f"Prompt: {prompts[j][:200]}...")
    print(f"Response: {responses[j]}")


## Rerun Notes

This rerun is intentionally deterministic: generation temperature is fixed at `0.0`. The earlier exploratory run used `0.7`; do not mix those numbers in the Week 5 handoff. The interpretation should be based on the upstream deterministic run.

Layer 20 remains the pre-registered primary layer, with 16 and 24 as ablations. The layer 13-24 sweep is exploratory and motivated by the Week 2/3 observation that the collusion signal is weak and appears earlier in the residual stream than expected.


In [0]:
# 9. Convert to numpy
acts_np = {layer: acts.numpy().astype(np.float32) for layer, acts in activations.items()}

PCA_COMPONENTS = 50

# --- Stratified 5-fold CV ---
print(f"--- Stratified 5-fold CV (PCA={PCA_COMPONENTS}) ---")
results_cv = {}
for layer in sorted(acts_np.keys()):
    cv = run_cv(acts_np[layer], labels, n_splits=5, pca_components=PCA_COMPONENTS)
    results_cv[layer] = cv
    print(f"  Layer {layer}: AUROC={cv.auroc_mean:.3f} +/- {cv.auroc_std:.3f}  "
          f"Acc={cv.accuracy_mean:.3f}  Brier={cv.brier_mean:.3f}  ECE={cv.ece_mean:.3f}")

In [0]:
# 10. Leave-scenario-out CV
print(f"--- Leave-scenario-out CV (PCA={PCA_COMPONENTS}) ---")
scenario_ids = np.array([m["scenario_idx"] for m in metadata])

results_lso = {}
for layer in sorted(acts_np.keys()):
    cv = run_leave_scenario_out(acts_np[layer], labels, scenario_ids, pca_components=PCA_COMPONENTS)
    results_lso[layer] = cv
    print(f"  Layer {layer}: AUROC={cv.auroc_mean:.3f} +/- {cv.auroc_std:.3f}  "
          f"Acc={cv.accuracy_mean:.3f}  Brier={cv.brier_mean:.3f}  ECE={cv.ece_mean:.3f}")

In [0]:
# 11. Deception directions and diff-in-means
print("--- Deception directions and diff-in-means ---")
directions = {}
for layer in sorted(acts_np.keys()):
    dec_dir, dec_norm = compute_deception_direction(acts_np[layer], labels)
    dim_dir, dim_norm = compute_diff_in_means(acts_np[layer], labels)
    cosine_sim = float(np.dot(dec_dir, dim_dir / dim_norm)) if dim_norm > 0 else 0.0
    directions[layer] = {"dec_norm": dec_norm, "dim_norm": dim_norm, "cosine_sim": cosine_sim}
    print(f"  Layer {layer}: ||deception dir||={dec_norm:.3f}  ||diff-in-means||={dim_norm:.3f}  "
          f"cosine(probe, DIM)={cosine_sim:.3f}")

In [0]:
# 12. Visualization: AUROC by layer (both CV methods)
import matplotlib.pyplot as plt

layers_sorted = sorted(results_cv.keys())
aurocs_cv = [results_cv[l].auroc_mean for l in layers_sorted]
stds_cv = [results_cv[l].auroc_std for l in layers_sorted]
aurocs_lso = [results_lso[l].auroc_mean for l in layers_sorted]
stds_lso = [results_lso[l].auroc_std for l in layers_sorted]

x = np.arange(len(layers_sorted))
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.bar(x - 0.15, aurocs_cv, 0.3, yerr=stds_cv, capsize=5, label="5-fold stratified CV", color="#2196F3")
ax.bar(x + 0.15, aurocs_lso, 0.3, yerr=stds_lso, capsize=5, label="Leave-scenario-out CV", color="#FF9800")
ax.axhline(y=0.5, color="gray", linestyle=":", label="Chance")
ax.set_xticks(x)
ax.set_xticklabels([str(l) for l in layers_sorted])
ax.set_xlabel("Layer")
ax.set_ylabel("AUROC")
ax.set_title("Collusion Detection: Colluder vs Honest (Llama 3.1 8B, PCA=50)")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [0]:
# 13. Visualization: Calibration metrics by layer
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

briers_cv = [results_cv[l].brier_mean for l in layers_sorted]
eces_cv = [results_cv[l].ece_mean for l in layers_sorted]
briers_lso = [results_lso[l].brier_mean for l in layers_sorted]
eces_lso = [results_lso[l].ece_mean for l in layers_sorted]

ax1.bar(x - 0.15, briers_cv, 0.3, label="5-fold CV", color="#2196F3")
ax1.bar(x + 0.15, briers_lso, 0.3, label="Leave-scenario-out", color="#FF9800")
ax1.set_xticks(x)
ax1.set_xticklabels([str(l) for l in layers_sorted])
ax1.set_xlabel("Layer")
ax1.set_ylabel("Brier Score")
ax1.set_title("Brier Score by Layer")
ax1.legend(fontsize=9)

ax2.bar(x - 0.15, eces_cv, 0.3, label="5-fold CV", color="#2196F3")
ax2.bar(x + 0.15, eces_lso, 0.3, label="Leave-scenario-out", color="#FF9800")
ax2.set_xticks(x)
ax2.set_xticklabels([str(l) for l in layers_sorted])
ax2.set_xlabel("Layer")
ax2.set_ylabel("ECE")
ax2.set_title("Expected Calibration Error by Layer")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [0]:
# 14. PCA visualization of activation geometry
from sklearn.decomposition import PCA as PCA_Viz

n = len(layers_sorted)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
for ax, layer in zip(axes, layers_sorted):
    pca = PCA_Viz(n_components=2)
    X_2d = pca.fit_transform(acts_np[layer])
    ev = pca.explained_variance_ratio_
    ax.scatter(X_2d[labels == 0, 0], X_2d[labels == 0, 1], alpha=0.6, s=30, label="Honest", c="#4CAF50")
    ax.scatter(X_2d[labels == 1, 0], X_2d[labels == 1, 1], alpha=0.6, s=30, label="Colluder", c="#F44336")
    ax.set_xlabel(f"PC1 ({ev[0]:.1%})")
    ax.set_ylabel(f"PC2 ({ev[1]:.1%})")
    ax.set_title(f"Layer {layer}")
    ax.legend(fontsize=8)
plt.suptitle("PCA of Residual Stream Activations (Colluder vs Honest)", y=1.02)
plt.tight_layout()
plt.show()

In [0]:
import json
import numpy as np
from datetime import datetime

# Heavy tensors go to a binary sidecar
np.savez_compressed(
    "week2_collusion_probe_activations.npz",
    **{f"layer_{l}": acts_np[l] for l in layers_sorted},
    labels=labels,
)

def cv_to_dict(cv):
    """Serialize a CVResult into JSON-safe primitives."""
    out = {
        "auroc_mean": float(cv.auroc_mean),
        "auroc_std": float(cv.auroc_std),
        "accuracy_mean": float(cv.accuracy_mean),
        "accuracy_std": float(cv.accuracy_std),
        "brier_mean": float(cv.brier_mean),
        "brier_std": float(cv.brier_std),
        "ece_mean": float(cv.ece_mean),
        "ece_std": float(cv.ece_std),
    }
    if hasattr(cv, "auroc_per_fold"):
        out["auroc_per_fold"] = [float(x) for x in cv.auroc_per_fold]
    if hasattr(cv, "per_fold"):
        out["per_fold"] = [
            {"auroc": float(f.auroc), "accuracy": float(f.accuracy),
             "brier": float(f.brier), "ece": float(f.ece)}
            for f in cv.per_fold
        ]
    return out


responses_file = "week2_collusion_probe_responses.json"
with open(responses_file, "w") as f:
    json.dump([
        {"prompt_index": i, "label": int(labels[i]), "metadata": metadata[i], "response": responses[i]}
        for i in range(len(responses))
    ], f, indent=2)

# Everything human-readable goes to JSON
output = {
    "experiment": "week2_collusion_probe",
    "model": MODEL_NAME,
    "date": datetime.now().isoformat(),
    "n_scenarios": len(SCENARIOS),
    "n_prompts": len(prompts),
    "n_colluder": int(labels.sum()),
    "n_honest": int((labels == 0).sum()),
    "token_position": "last_generated",
    "max_new_tokens": MAX_NEW_TOKENS,
    "generation_temperature": GENERATION_TEMPERATURE,
    "top_p": TOP_P,
    "layers": list(DEFAULT_LAYERS),
    "source_dataset": "NARCBench-Core upstream generation/core.py",
    "pca_components": PCA_COMPONENTS,
    "activations_file": "week2_collusion_probe_activations.npz",
    "responses_file": responses_file,
    "stratified_cv": {},
    "leave_scenario_out_cv": {},
    "directions": {},
    "labels": labels.tolist(),
}

for layer in layers_sorted:
    output["stratified_cv"][str(layer)] = cv_to_dict(results_cv[layer])
    output["leave_scenario_out_cv"][str(layer)] = cv_to_dict(results_lso[layer])

    dec_dir, dec_norm = compute_deception_direction(acts_np[layer], labels)
    dim_dir, dim_norm = compute_diff_in_means(acts_np[layer], labels)
    output["directions"][str(layer)] = {
        "probe_direction": dec_dir.tolist(),
        "dim_direction": dim_dir.tolist(),
        "probe_norm": float(dec_norm),
        "dim_norm": float(dim_norm),
        "cosine_sim": float(directions[layer]["cosine_sim"]),
    }

with open("week2_collusion_probe_results.json", "w") as f:
    json.dump(output, f, indent=2)

print("Results saved to week2_collusion_probe_results.json")
print("Activations saved to week2_collusion_probe_activations.npz")
print(f"Responses saved to {responses_file}")


In [0]:
print(vars(results_cv[layers_sorted[0]]))

In [0]:
# 16. Summary table
print(f"{'Layer':<8} {'AUROC (5-fold)':<20} {'AUROC (LSO)':<20} {'Brier (5f)':<12} {'ECE (5f)':<10} {'cos(probe,DIM)':<15}")
print("-" * 85)
for layer in layers_sorted:
    cv = results_cv[layer]
    lso = results_lso[layer]
    d = directions[layer]
    print(f"{layer:<8} {cv.auroc_mean:.3f} +/- {cv.auroc_std:.3f}      "
          f"{lso.auroc_mean:.3f} +/- {lso.auroc_std:.3f}      "
          f"{cv.brier_mean:.3f}       {cv.ece_mean:.3f}     {d['cosine_sim']:.3f}")

In [0]:
# Install EP github ( a bit out of scope but a new way to look inside models and perhaps cheaper?)
!pip install -q git+https://github.com/jessicarumbelow/exemplar-partitioning.git

## Exemplar Partitioning (Rumbelow et al. 2025)

Unsupervised directional clustering of residual stream activations via leader clustering. Runs on already-extracted activations, with no extra forward passes.

Layers: 13 (exploratory early layer), 16 (committed ablation), 20 (pre-registered primary), 24 (late-layer ablation).

Goal: check whether internal activation geometry separates collusion from honesty without labels. If EP partitions correlate with collusion labels, the signal exists independently of the trained probe. Interpret this only after the deterministic rerun; do not import conclusions from earlier exploratory notes.


In [0]:
# 17. EP: Calibrate + build dictionaries across layers
from ep.discovery.calibration import Calibration, calibrate
from ep.discovery.dictionary import Dictionary, _to_directions
from scipy.stats import fisher_exact
import numpy as np

EP_LAYERS = [13, 16, 20, 24]
EP_PERCENTILE = 10.0
BATCH_SIZE = 20

ep_results = {}

for layer in EP_LAYERS:
    X_ep = acts_np[layer]

    cal = calibrate(
        [X_ep[i:i+BATCH_SIZE] for i in range(0, len(X_ep), BATCH_SIZE)],
        n_tokens=len(X_ep),
        percentile=EP_PERCENTILE,
    )

    ep_dict = Dictionary(center=cal.center, threshold=cal.threshold)
    ep_dict.add_batch(X_ep, iteration=0, global_index_start=0)
    ep_dict.finalize()

    ep_dirs = _to_directions(X_ep.astype(np.float32), cal.center)
    exemplars = np.stack([p.exemplar_direction for p in ep_dict.partitions])
    sims = ep_dirs @ exemplars.T
    assignments = sims.argmax(axis=1)

    probe_dir, probe_norm = compute_deception_direction(X_ep, labels)
    dim_dir, dim_norm = compute_diff_in_means(X_ep, labels)

    ep_results[layer] = {
        "calibration": cal,
        "dictionary": ep_dict,
        "ep_directions": ep_dirs,
        "exemplars": exemplars,
        "assignments": assignments,
        "probe_dir": probe_dir,
        "probe_norm": probe_norm,
        "dim_dir": dim_dir,
        "dim_norm": dim_norm,
    }

    print(f"Layer {layer}: {len(ep_dict.partitions)} partitions, "
          f"threshold={cal.threshold:.4f}, "
          f"center_norm={np.linalg.norm(cal.center):.2f}")


In [0]:
# 18. EP: Partition enrichment analysis (Fisher's exact test per partition)
for layer in EP_LAYERS:
    r = ep_results[layer]
    ep_dict = r["dictionary"]
    assignments = r["assignments"]

    print(f"\n{'='*65}")
    print(f"Layer {layer}: {len(ep_dict.partitions)} partitions")
    print(f"{'='*65}")
    print(f"{'PID':<6} {'Size':<6} {'Colluder%':<10} {'Coherence':<10} {'p (Fisher)':<10} {'Sig':<4}")
    print("-" * 50)

    enriched = []
    for pid, p in enumerate(ep_dict.partitions):
        mask = assignments == pid
        n = int(mask.sum())
        if n == 0:
            continue
        n_col = int(labels[mask].sum())
        n_hon = n - n_col
        rate = n_col / n

        rest = ~mask
        n_col_rest = int(labels[rest].sum())
        n_hon_rest = int((labels[rest] == 0).sum())
        table = [[n_col, n_hon], [n_col_rest, n_hon_rest]]
        _, pval = fisher_exact(table)

        sig = "*" if pval < 0.05 else ("+" if pval < 0.10 else "")
        enriched.append({
            "pid": pid, "n": n, "rate": rate,
            "coherence": p.member_coherence, "pval": pval,
        })
        print(f"{pid:<6} {n:<6} {rate:<10.2f} {p.member_coherence:<10.3f} {pval:<10.3f} {sig}")

    col_enriched = [e for e in enriched if e["rate"] > 0.7]
    hon_enriched = [e for e in enriched if e["rate"] < 0.3]

    if col_enriched:
        print(f"\nCollusion-enriched (>70%):")
        for e in col_enriched:
            print(f"  Partition {e['pid']}: {e['n']} members, "
                  f"{e['rate']:.0%} colluder, p={e['pval']:.3f}")
    if hon_enriched:
        print(f"\nHonesty-enriched (<30%):")
        for e in hon_enriched:
            print(f"  Partition {e['pid']}: {e['n']} members, "
                  f"{e['rate']:.0%} colluder, p={e['pval']:.3f}")
    if not col_enriched and not hon_enriched:
        print(f"\nNo partitions exceeded 70%/30% enrichment thresholds.")

    ep_results[layer]["enriched"] = enriched

In [0]:
# 19. EP: Exemplar-to-probe direction alignment
for layer in EP_LAYERS:
    r = ep_results[layer]
    ep_dict = r["dictionary"]
    probe_dir = r["probe_dir"]
    dim_dir = r["dim_dir"]
    enriched = r["enriched"]

    print(f"\nLayer {layer}:")
    print(f"{'PID':<6} {'Colluder%':<10} {'cos(ex,probe)':<16} {'cos(ex,DIM)':<14} {'Size':<6} {'p':<8}")
    print("-" * 60)

    for e in sorted(enriched, key=lambda x: -x["rate"]):
        ex = ep_dict.partitions[e["pid"]].exemplar_direction
        ex_norm = np.linalg.norm(ex)
        if ex_norm == 0:
            continue
        cos_probe = float(np.dot(ex, probe_dir) / (ex_norm * np.linalg.norm(probe_dir)))
        cos_dim = float(np.dot(ex, dim_dir) / (ex_norm * np.linalg.norm(dim_dir)))
        marker = " <--" if e["rate"] > 0.7 or e["rate"] < 0.3 else ""
        print(f"{e['pid']:<6} {e['rate']:<10.2f} {cos_probe:<16.3f} {cos_dim:<14.3f} "
              f"{e['n']:<6} {e['pval']:<8.3f}{marker}")

In [0]:
# 20. EP: Multi-layer PCA visualization + cross-layer summary
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for ax, layer in zip(axes.flat, EP_LAYERS):
    r = ep_results[layer]
    X_ep = acts_np[layer]

    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X_ep)
    ev = pca.explained_variance_ratio_

    scatter = ax.scatter(
        X_2d[:, 0], X_2d[:, 1],
        c=r["assignments"], cmap="tab20",
        alpha=0.5, s=30, edgecolors="none",
    )
    ax.scatter(
        X_2d[labels == 0, 0], X_2d[labels == 0, 1],
        marker="o", facecolors="none", edgecolors="#4CAF50",
        s=70, linewidths=1.2, label="Honest",
    )
    ax.scatter(
        X_2d[labels == 1, 0], X_2d[labels == 1, 1],
        marker="s", facecolors="none", edgecolors="#F44336",
        s=70, linewidths=1.2, label="Colluder",
    )
    n_parts = len(r["dictionary"].partitions)
    n_enriched = sum(1 for e in r["enriched"] if e["rate"] > 0.7 or e["rate"] < 0.3)
    ax.set_xlabel(f"PC1 ({ev[0]:.1%})")
    ax.set_ylabel(f"PC2 ({ev[1]:.1%})")
    ax.set_title(f"Layer {layer}: {n_parts} partitions, {n_enriched} enriched")
    ax.legend(fontsize=8)

plt.suptitle("EP Partitions (color) vs Collusion Labels (shape)", y=1.01)
plt.tight_layout()
plt.savefig("week2_ep_partitions_multilayer.png", dpi=150, bbox_inches="tight")
plt.show()

# Cross-layer summary table
print(f"\n{'Layer':<7} {'Parts':<7} {'Enriched':<10} {'Max col%':<10} {'Min col%':<10} "
      f"{'Best p':<10} {'Threshold':<10}")
print("-" * 64)
for layer in EP_LAYERS:
    r = ep_results[layer]
    enriched = r["enriched"]
    n_parts = len(r["dictionary"].partitions)
    rates = [e["rate"] for e in enriched]
    pvals = [e["pval"] for e in enriched]
    n_enriched = sum(1 for e in enriched if e["rate"] > 0.7 or e["rate"] < 0.3)
    print(f"{layer:<7} {n_parts:<7} {n_enriched:<10} {max(rates):<10.2f} {min(rates):<10.2f} "
          f"{min(pvals):<10.3f} {r['calibration'].threshold:<10.4f}")

# Partition size distributions
fig, axes = plt.subplots(1, len(EP_LAYERS), figsize=(4 * len(EP_LAYERS), 3))
for ax, layer in zip(axes, EP_LAYERS):
    r = ep_results[layer]
    enriched = r["enriched"]
    sizes = [e["n"] for e in enriched]
    rates = [e["rate"] for e in enriched]
    colors = ["#F44336" if rate > 0.7 else "#4CAF50" if rate < 0.3 else "#9E9E9E"
              for rate in rates]
    ax.bar(range(len(sizes)), sizes, color=colors)
    ax.set_xlabel("Partition ID")
    ax.set_ylabel("Size")
    ax.set_title(f"Layer {layer}")
plt.suptitle("Partition sizes (red=collusion-enriched, green=honesty-enriched, grey=mixed)")
plt.tight_layout()
plt.savefig("week2_ep_partition_sizes.png", dpi=150, bbox_inches="tight")
plt.show()

In [0]:
# 21. Save EP results
import json

ep_output = {
    "method": "exemplar_partitioning",
    "reference": "Rumbelow et al. 2025",
    "percentile": EP_PERCENTILE,
    "layers": {},
}

for layer in EP_LAYERS:
    r = ep_results[layer]
    ep_output["layers"][str(layer)] = {
        "n_partitions": len(r["dictionary"].partitions),
        "threshold": float(r["calibration"].threshold),
        "center_norm": float(np.linalg.norm(r["calibration"].center)),
        "partitions": [],
    }
    for e in r["enriched"]:
        ex = r["dictionary"].partitions[e["pid"]].exemplar_direction
        ex_norm = float(np.linalg.norm(ex))
        cos_probe = float(np.dot(ex, r["probe_dir"]) / (ex_norm * np.linalg.norm(r["probe_dir"]))) if ex_norm > 0 else 0.0
        cos_dim = float(np.dot(ex, r["dim_dir"]) / (ex_norm * np.linalg.norm(r["dim_dir"]))) if ex_norm > 0 else 0.0
        ep_output["layers"][str(layer)]["partitions"].append({
            "pid": e["pid"],
            "size": e["n"],
            "collusion_rate": round(e["rate"], 4),
            "coherence": round(e["coherence"], 4),
            "fisher_p": round(e["pval"], 6),
            "cos_probe": round(cos_probe, 4),
            "cos_dim": round(cos_dim, 4),
        })

with open("week2_ep_results.json", "w") as f:
    json.dump(ep_output, f, indent=2)
print("EP results saved to week2_ep_results.json")
print(json.dumps(ep_output, indent=2))

In [0]:
# Save generated artifacts to the shared Drive artifact directory
import json
import shutil
from datetime import datetime
from pathlib import Path

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILES = [
    "week2_collusion_probe_results.json",
    "week2_collusion_probe_activations.npz",
    "week2_collusion_probe_responses.json",
    "week2_ep_results.json",
    "week2_ep_partitions_multilayer.png",
    "week2_ep_partition_sizes.png",
]

saved = []
missing = []
for fname in OUTPUT_FILES:
    src = Path(fname)
    if src.exists():
        dst = ARTIFACT_DIR / fname
        shutil.copy2(src, dst)
        saved.append(str(dst))
        print(f"Saved {fname} -> {dst}")
    else:
        missing.append(fname)
        print(f"Skipped missing artifact: {fname}")

manifest = {
    "notebook": "02_collusion_probe.ipynb",
    "created_at": datetime.now().isoformat(),
    "artifact_dir": str(ARTIFACT_DIR),
    "saved_files": saved,
    "missing_files": missing,
    "next_notebook": "03_analysis.ipynb",
}
with open(ARTIFACT_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Manifest saved to {ARTIFACT_DIR / 'manifest.json'}")
